In [ ]:


# ============================================================
# PERSONAL AI TASK AGENT - COMPLETE GOOGLE COLAB PROJECT
# Run this entire code in ONE Google Colab cell.
# ============================================================

%pip -q install openai ipywidgets

import os
import json
import sqlite3
from datetime import date, timedelta
from getpass import getpass
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

# -------------------- DATABASE --------------------

DB = "/content/personal_ai_tasks.db"

def init_db():
    with sqlite3.connect(DB) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS tasks (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                title TEXT NOT NULL,
                description TEXT DEFAULT '',
                due_date TEXT NOT NULL,
                priority TEXT DEFAULT 'Medium',
                status TEXT DEFAULT 'Pending'
            )
        """)

def add_task(title, description, due_date, priority):
    with sqlite3.connect(DB) as conn:
        conn.execute("""
            INSERT INTO tasks
            (title, description, due_date, priority, status)
            VALUES (?, ?, ?, ?, 'Pending')
        """, (title, description, str(due_date), priority))

def get_tasks():
    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(r) for r in conn.execute(
            "SELECT * FROM tasks ORDER BY due_date, id"
        ).fetchall()]

def update_task(task_id, status):
    with sqlite3.connect(DB) as conn:
        conn.execute(
            "UPDATE tasks SET status=? WHERE id=?",
            (status, task_id)
        )

def delete_task(task_id):
    with sqlite3.connect(DB) as conn:
        conn.execute("DELETE FROM tasks WHERE id=?", (task_id,))

init_db()

# -------------------- GENERATIVE AI --------------------

def generate_plan(goal, days, hours, api_key):
    from openai import OpenAI

    client = OpenAI(api_key=api_key)

    response = client.responses.create(
        model="gpt-4.1-mini",
        instructions="""
        You are a Personal AI Task Planning Agent.
        Return only a valid JSON array of 3 to 10 tasks.
        Every task must contain:
        title, description, days_from_today, priority.
        days_from_today must be an integer from 0 to days-1.
        priority must be High, Medium, or Low.
        Create practical, achievable tasks.
        """,
        input=(
            f"Goal: {goal}\n"
            f"Available days: {days}\n"
            f"Hours available per day: {hours}\n"
            f"Today: {date.today()}"
        )
    )

    raw = response.output_text.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    plan = json.loads(raw)

    if not isinstance(plan, list) or not 3 <= len(plan) <= 10:
        raise ValueError("The AI returned an invalid plan.")

    for task in plan:
        if not isinstance(task, dict):
            raise ValueError("Invalid task format.")

        title = task.get("title")
        description = task.get("description", "")
        offset = task.get("days_from_today")
        priority = task.get("priority")

        if not isinstance(title, str) or not title.strip():
            raise ValueError("Missing task title.")
        if not isinstance(description, str):
            raise ValueError("Invalid task description.")
        if type(offset) is not int or not 0 <= offset < days:
            raise ValueError("Invalid task date.")
        if priority not in ("High", "Medium", "Low"):
            raise ValueError("Invalid task priority.")

    return plan

# -------------------- DASHBOARD --------------------

dashboard = widgets.Output()
message = widgets.Output()

def refresh_dashboard():
    tasks = get_tasks()
    completed = sum(t["status"] == "Completed" for t in tasks)
    pending = len(tasks) - completed
    progress = round(100 * completed / len(tasks)) if tasks else 0

    with dashboard:
        clear_output(wait=True)

        display(Markdown("## 📊 Task Dashboard"))
        display(Markdown(
            f"**Total:** {len(tasks)} | "
            f"**Completed:** {completed} | "
            f"**Pending:** {pending} | "
            f"**Progress:** {progress}%"
        ))

        if not tasks:
            display(Markdown("No tasks yet. Add a task or generate an AI plan."))
            return

        for task in tasks:
            icon = "✅" if task["status"] == "Completed" else "⏳"
            display(Markdown(
                f"### {icon} {task['id']}. {task['title']}\n"
                f"- Description: {task['description']}\n"
                f"- Due date: {task['due_date']}\n"
                f"- Priority: {task['priority']}\n"
                f"- Status: {task['status']}"
            ))

def show_message(text):
    with message:
        clear_output(wait=True)
        display(Markdown(text))

# -------------------- MANUAL TASK FORM --------------------

task_title = widgets.Text(
    description="Task:",
    placeholder="Study Python",
    layout=widgets.Layout(width="95%")
)
task_description = widgets.Textarea(
    description="Details:",
    placeholder="Task description",
    layout=widgets.Layout(width="95%")
)
task_date = widgets.DatePicker(
    description="Due date:",
    value=date.today() + timedelta(days=1)
)
task_priority = widgets.Dropdown(
    description="Priority:",
    options=["High", "Medium", "Low"],
    value="Medium"
)

def on_add_task(_):
    if not task_title.value.strip():
        show_message("⚠️ Please enter a task title.")
        return

    if task_date.value is None:
        show_message("⚠️ Please select a due date.")
        return

    add_task(
        task_title.value.strip(),
        task_description.value.strip(),
        task_date.value,
        task_priority.value
    )

    task_title.value = ""
    task_description.value = ""
    show_message("✅ Task added successfully.")
    refresh_dashboard()

add_button = widgets.Button(
    description="Add Task",
    button_style="success",
    icon="plus"
)
add_button.on_click(on_add_task)

# -------------------- AI PLANNER FORM --------------------

goal_input = widgets.Textarea(
    description="Goal:",
    placeholder="Example: Prepare for my Python exam",
    layout=widgets.Layout(width="95%", height="85px")
)
days_input = widgets.BoundedIntText(
    description="Days:",
    value=5,
    min=1,
    max=365
)
hours_input = widgets.BoundedIntText(
    description="Hours/day:",
    value=2,
    min=1,
    max=24
)

api_input = widgets.Password(
    description="API key:",
    placeholder="Enter OpenAI API key for AI planning",
    layout=widgets.Layout(width="95%")
)

plan_button = widgets.Button(
    description="Generate AI Plan",
    button_style="primary",
    icon="magic"
)

def on_generate_plan(_):
    goal = goal_input.value.strip()
    key = api_input.value.strip()

    if not goal:
        show_message("⚠️ Enter your goal first.")
        return

    if not key:
        show_message(
            "⚠️ Enter your OpenAI API key to use AI planning. "
            "Manual task management works without it."
        )
        return

    plan_button.disabled = True
    show_message("⏳ AI is preparing your plan...")

    try:
        plan = generate_plan(
            goal, days_input.value, hours_input.value, key
        )

        # Preview first; only save after user confirmation.
        preview_output.clear_output(wait=True)
        with preview_output:
            display(Markdown("## 📝 AI Plan Preview"))
            for i, task in enumerate(plan, 1):
                due = date.today() + timedelta(
                    days=task["days_from_today"]
                )
                display(Markdown(
                    f"**{i}. {task['title']}** — {due}\n\n"
                    f"{task['description']}\n\n"
                    f"Priority: {task['priority']}"
                ))

        pending_plan.clear()
        pending_plan.extend(plan)
        save_plan_button.disabled = False
        show_message("Review the plan below, then click Save AI Plan.")

    except Exception as e:
        show_message(f"❌ Could not generate the plan: {e}")

    finally:
        plan_button.disabled = False

plan_button.on_click(on_generate_plan)

# -------------------- SAVE GENERATED PLAN --------------------

pending_plan = []

preview_output = widgets.Output()

save_plan_button = widgets.Button(
    description="Save AI Plan to Tasks",
    button_style="success",
    icon="save",
    disabled=True
)

def on_save_plan(_):
    if not pending_plan:
        show_message("⚠️ Generate a plan first.")
        return

    for task in pending_plan:
        due = date.today() + timedelta(
            days=task["days_from_today"]
        )
        add_task(
            task["title"].strip(),
            task.get("description", ""),
            due,
            task["priority"]
        )

    count = len(pending_plan)
    pending_plan.clear()
    save_plan_button.disabled = True

    with preview_output:
        clear_output(wait=True)

    show_message(f"✅ Saved {count} AI-generated tasks.")
    refresh_dashboard()

save_plan_button.on_click(on_save_plan)

# -------------------- UPDATE OR DELETE TASK --------------------

tasks_dropdown = widgets.Dropdown(
    description="Task ID:",
    options=[],
    layout=widgets.Layout(width="95%")
)

status_dropdown = widgets.Dropdown(
    description="Status:",
    options=["Pending", "In Progress", "Completed"],
    value="Completed"
)

def refresh_task_options():
    tasks = get_tasks()
    tasks_dropdown.options = [
        (f"{t['id']} - {t['title']}", t["id"])
        for t in tasks
    ]

def on_update_task(_):
    if tasks_dropdown.value is None:
        show_message("⚠️ No task selected.")
        return

    update_task(tasks_dropdown.value, status_dropdown.value)
    show_message("✅ Task status updated.")
    refresh_dashboard()
    refresh_task_options()

def on_delete_task(_):
    if tasks_dropdown.value is None:
        show_message("⚠️ No task selected.")
        return

    delete_task(tasks_dropdown.value)
    show_message("🗑️ Task deleted.")
    refresh_dashboard()
    refresh_task_options()

update_button = widgets.Button(
    description="Update Status",
    button_style="info"
)
update_button.on_click(on_update_task)

delete_button = widgets.Button(
    description="Delete Task",
    button_style="danger",
    icon="trash"
)
delete_button.on_click(on_delete_task)

# -------------------- DISPLAY APP --------------------

display(Markdown("# 🤖 Personal AI Task Agent"))
display(Markdown(
    "Manage your daily activities, generate AI plans, "
    "and monitor your progress."
))

display(widgets.HTML("<h3>1. Add a Task Manually</h3>"))
display(task_title, task_description, task_date, task_priority, add_button)

display(widgets.HTML("<hr><h3>2. Generate an AI Task Plan</h3>"))
display(goal_input, days_input, hours_input, api_input, plan_button)
display(preview_output, save_plan_button)

display(widgets.HTML("<hr><h3>3. Update or Delete Tasks</h3>"))
refresh_task_options()
display(tasks_dropdown, status_dropdown, update_button, delete_button)

display(widgets.HTML("<hr>"))
display(message)
display(dashboard)

refresh_dashboard()

print("Personal AI Task Agent is ready!")